# Quantum Randomness for Strong Passwords and Keys

This notebook follows the companion blog post. It simulates an ideal measurement of a qubit prepared in the Hadamard state and uses the resulting bits to build passwords without modulo bias.

Requirements: Python 3.10 or newer. No third-party packages are needed.

## 1. A qubit as a fair quantum coin

Starting from `|0>`, applying a Hadamard gate gives

`|+> = (|0> + |1>) / sqrt(2)`.

When measured in the computational basis, the probabilities are 50% for `0` and 50% for `1`.

In [ ]:
import math
import secrets
import string
from collections import Counter

def measure_hadamard_qubit():
    amplitudes = (1 / math.sqrt(2), 1 / math.sqrt(2))
    probabilities = tuple(round(abs(amplitude) ** 2, 10) for amplitude in amplitudes)
    assert probabilities == (0.5, 0.5)
    return secrets.randbelow(2)

def quantum_bits(length):
    return [measure_hadamard_qubit() for _ in range(length)]

bits = quantum_bits(32)
''.join(str(bit) for bit in bits)

## 2. Check the balance

A real QRNG would need careful hardware calibration and statistical testing. For a beginner simulation, a simple monobit balance check is a useful first sanity check.

In [ ]:
def monobit_report(bits):
    counts = Counter(bits)
    zeros = counts[0]
    ones = counts[1]
    total = zeros + ones
    z_score = (ones - zeros) / math.sqrt(total)
    return zeros, ones, z_score

sample = quantum_bits(1024)
monobit_report(sample)

## 3. Turn bits into a password without bias

If an alphabet has 74 symbols, `value % 74` is biased because powers of two do not divide evenly by 74. Rejection sampling avoids that problem.

In [ ]:
ALPHABET = string.ascii_letters + string.digits + "!@#$%^&*()-_=+"

def bits_to_int(bits):
    value = 0
    for bit in bits:
        value = (value << 1) | bit
    return value

def sample_alphabet_index(alphabet_size):
    bits_needed = math.ceil(math.log2(alphabet_size))
    while True:
        candidate = bits_to_int(quantum_bits(bits_needed))
        if candidate < alphabet_size:
            return candidate

def quantum_password(length=16, alphabet=ALPHABET):
    return ''.join(alphabet[sample_alphabet_index(len(alphabet))] for _ in range(length))

quantum_password(16)

## 4. A toy one-time-pad example

A one-time pad needs a truly random key that is at least as long as the message and never reused. This cell shows the reversible XOR step on a short message.

In [ ]:
def quantum_key_bytes(length):
    return bytes(bits_to_int(quantum_bits(8)) for _ in range(length))

def xor_bytes(message, key):
    if len(key) < len(message):
        raise ValueError("key must be at least as long as the message")
    return bytes(message_byte ^ key_byte for message_byte, key_byte in zip(message, key))

message = b"meet at 5"
key = quantum_key_bytes(len(message))
ciphertext = xor_bytes(message, key)
recovered = xor_bytes(ciphertext, key)

message, key.hex(), ciphertext.hex(), recovered